In [10]:
!pip install pdfplumber google-generativeai

In [11]:
import os
from google.colab import userdata
import google.generativeai as genai

# Safely retrieve your secret key from Colab's secrets tab
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

# Configure the Gemini library with your key
genai.configure(api_key=GOOGLE_API_KEY)
print("Gemini API Key successfully loaded!")

Gemini API Key successfully loaded!


In [12]:
import os

# Create the required folders for the assignment
os.makedirs("invoices", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("Folders created! Click the 'Folder' icon on the left sidebar to see 'invoices' and 'outputs'.")
print("👉 Please upload your 3 invoice PDFs into the 'invoices' folder now.")

Folders created! Click the 'Folder' icon on the left sidebar to see 'invoices' and 'outputs'.
👉 Please upload your 3 invoice PDFs into the 'invoices' folder now.


In [14]:
import json
import pdfplumber
import google.generativeai as genai

def extract_invoice_data(pdf_path):
    """
    Complete pipeline to extract structured JSON data from an invoice PDF using Gemini.
    """
    print(f"--- Starting Pipeline for: {pdf_path} ---")

    # ------------------------------------------------------------------------
    # PIPELINE STEP 1 & 2: Load the PDF and Extract Raw Text
    # ------------------------------------------------------------------------
    raw_text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:  # Step 1: Open the invoice PDF file
            for page in pdf.pages:
                text = page.extract_text()      # Step 2: Loop through pages and grab text
                if text:
                    raw_text += text + "\n"
    except Exception as e:
        print(f"Error reading PDF file: {e}")
        return None

    if not raw_text.strip():
        print("Error: Extracted text is empty. Check your PDF file format.")
        return None

    # ------------------------------------------------------------------------
    # PIPELINE STEP 3: Clean the Text
    # ------------------------------------------------------------------------
    # Remove extra empty spaces and trailing lines so the data is tidy
    lines = [line.strip() for line in raw_text.split("\n")]
    cleaned_text = "\n".join([line for line in lines if line])

    # ------------------------------------------------------------------------
    # PIPELINE STEP 4 & 5: Setup Prompt and Call Gemini
    # ------------------------------------------------------------------------
    # We use 'gemini-2.5-flash' at temperature 0 for fast, structured data extraction
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        generation_config={
            "temperature": 0,  # Forces deterministic, non-guessed answers
            "response_mime_type": "application/json"  # Enforces clean JSON structure natively!
        }
    )

    # This prompt tells Gemini to extract all 13 mandatory assignment fields
    prompt = f"""
    You are an expert data extraction AI specialized in financial invoices.
    Analyze the raw invoice text provided below and extract key details into a structured JSON format.

    You MUST extract exactly these 13 fields:
    1. invoice_number: The unique invoice ID or reference number.
    2. invoice_date: Date the invoice was issued — strictly formatted as DD-MM-YYYY.
    3. due_date: Payment deadline — format as DD-MM-YYYY or descriptive string if formatted that way (e.g., 'Upon Receipt').
    4. billed_by: Name of the company or person issuing the invoice.
    5. billed_to: Name of the client or recipient being billed.
    6. line_items: A list of objects, where each object contains exactly {{"item": "item name", "amount": "item amount"}}.
    7. subtotal: Total amount before any discounts or taxes.
    8. discount: Any discount applied.
    9. tax_or_gst: Tax or GST amount if mentioned.
    10. total_amount: The final payable amount including all taxes/discounts.
    11. currency: Currency used in the invoice (e.g., INR, USD).
    12. payment_method: How payment should be made (e.g., NEFT, UPI, Credit Card).
    13. notes: Any additional remarks, instructions, or terms on the invoice.

    CRITICAL RULES:
    - If a field is not present in the invoice, set its value to null. Never guess, invent data, or use placeholder strings like 'N/A'.
    - Return ONLY the raw JSON string starting with {{ and ending with }}.

    Invoice Text:
    {cleaned_text}
    """

    # ------------------------------------------------------------------------
    # PIPELINE STEP 6: Run API and Parse the JSON
    # ------------------------------------------------------------------------
    try:
        response = model.generate_content(prompt)
        raw_response = response.text.strip()

        # Convert text returned by Gemini into a readable Python dictionary
        extracted_json = json.loads(raw_response)
        return extracted_json
    except Exception as e:
        print(f"Gemini API call or JSON parsing failed: {e}")
        return None

In [15]:
# PIPELINE STEP 7: Display and Run the Pipeline
invoice_files = [
    ("invoices/invoice_1.pdf", "outputs/output_invoice_1.json"),
    ("invoices/invoice_2.pdf", "outputs/output_invoice_2.json"),
    ("invoices/invoice_3.pdf", "outputs/output_invoice_3.json")
]

for pdf_in, json_out in invoice_files:
    if os.path.exists(pdf_in):
        # Run our Gemini pipeline
        result = extract_invoice_data(pdf_in)

        if result:
            # Print each extracted field clearly to your Colab screen
            print("\nSuccessfully Extracted Fields:")
            print(json.dumps(result, indent=4, ensure_ascii=False))

            # Save the JSON data directly into the outputs/ folder
            with open(json_out, "w", encoding="utf-8") as f:
                json.dump(result, f, indent=4, ensure_ascii=False)
            print(f"Saved JSON data to {json_out}\n")
    else:
        print(f"Skipping: '{pdf_in}' not found. Please make sure files are uploaded inside the 'invoices' folder.")

    print("-" * 50)

--- Starting Pipeline for: invoices/invoice_1.pdf ---

Successfully Extracted Fields:
{
    "invoice_number": "CTO0001",
    "invoice_date": "01-01-2021",
    "due_date": null,
    "billed_by": "Defmacro Software Dilip Merugu Put. Ltd",
    "billed_to": null,
    "line_items": [
        {
            "item": "UI/UN Design for eb & nobile phase",
            "amount": "1200.0"
        },
        {
            "item": "UI/UX Design for web nob1le phase",
            "amount": "1200.0e"
        }
    ],
    "subtotal": "10,000.00",
    "discount": null,
    "tax_or_gst": "4,880.00",
    "total_amount": "14,880.00",
    "currency": "INR",
    "payment_method": "UPI",
    "notes": "It was a pleasure doing business with you. Thank you! Clear One."
}
Saved JSON data to outputs/output_invoice_1.json

--------------------------------------------------
--- Starting Pipeline for: invoices/invoice_2.pdf ---


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 11886.87ms



Successfully Extracted Fields:
{
    "invoice_number": "X-7",
    "invoice_date": "21-06-2018",
    "due_date": "28-06-2018",
    "billed_by": "SC Aissac SRL",
    "billed_to": "Test Client US",
    "line_items": [
        {
            "item": "Mouse Z3 44mm, optical, black",
            "amount": "1319.07"
        },
        {
            "item": "Shipping & Packaging charges",
            "amount": "17.59"
        }
    ],
    "subtotal": null,
    "discount": null,
    "tax_or_gst": null,
    "total_amount": "1336.66",
    "currency": "USD",
    "payment_method": null,
    "notes": "To be paid in full in maximum 7 days after receiving the invoice. Warranty: 30 days from receiving items. Supply meant for export on payment of integrated tax."
}
Saved JSON data to outputs/output_invoice_2.json

--------------------------------------------------
--- Starting Pipeline for: invoices/invoice_3.pdf ---


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3413.01ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2023.06ms



Successfully Extracted Fields:
{
    "invoice_number": "6",
    "invoice_date": "13-09-2018",
    "due_date": "13-10-2018",
    "billed_by": "Alpha Computer",
    "billed_to": "Alan TEST",
    "line_items": [
        {
            "item": "Backpack Bags",
            "amount": "11,186.40"
        },
        {
            "item": "hp All ín one 20-comDl",
            "amount": "24,152.54"
        },
        {
            "item": "Hp Headphones BABOS Wired",
            "amount": "2,203.36"
        },
        {
            "item": "shipping & Packaging",
            "amount": "236.00"
        }
    ],
    "subtotal": "37,778.30",
    "discount": null,
    "tax_or_gst": null,
    "total_amount": "37,778",
    "currency": "INR",
    "payment_method": "Bank Transfer",
    "notes": "Rounded off by 0.30. Supply meant for export under bond or letter of undertaking without payment of integrated tax."
}
Saved JSON data to outputs/output_invoice_3.json

------------------------------------------

In [16]:
with open("requirements.txt", "w") as req_file:
    req_file.write("pdfplumber\ngoogle-generativeai\n")
print("requirements.txt generated successfully!")

requirements.txt generated successfully!
